#### Roteamentos

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import Ollama

In [3]:
chat_models = Ollama(model="minimax-m2.5:cloud", temperature=.1)

/tmp/ipykernel_183824/2804026812.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  chat_models = Ollama(model="minimax-m2.5:cloud", temperature=.1)


In [4]:
chat_models.invoke("Ola")


"\n\nHello! 👋\n\nHow can I help you today? I'm here to assist with coding, explanations, projects, or any questions you have."

##### Vamos tentar simular o roteamento de pergunta para assunto expecificos

In [5]:
prompt_base = "Você é um professor da materia de {materia} do ensino médio e é capaz de dar respostas clara e detalhada. Responda o seguinte para o aluno a duvida: {duvida}"

chat_matematica = ChatPromptTemplate.from_template(prompt_base.format(materia="matematica", duvida="Como é feito o calculo de pitagoras?"))
chain_matematica = chat_matematica | chat_models

chat_portugues = ChatPromptTemplate.from_template(prompt_base.format(materia="portugues",duvida="O que é um objeto inanimado?"))
chain_portugues = chat_portugues | chat_models

chat_biologia = ChatPromptTemplate.from_template(prompt_base.format(materia="biologia", duvida="Quantos animais mamiferos existem?"))
chain_biologia = chat_biologia | chat_models

chat_generico = ChatPromptTemplate.from_template("Essa questão é muito generica!!! {duvida}")
chain_generico = chat_generico | chat_models

print(chain_matematica)
print(chain_portugues)
print(chain_biologia)
print(chain_generico)

first=ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Você é um professor da materia de matematica do ensino médio e é capaz de dar respostas clara e detalhada. Responda o seguinte para o aluno a duvida: Como é feito o calculo de pitagoras?'), additional_kwargs={})]) middle=[] last=Ollama(model='minimax-m2.5:cloud', temperature=0.1)
first=ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Você é um professor da materia de portugues do ensino médio e é capaz de dar respostas clara e detalhada. Responda o seguinte para o aluno a duvida: O que é um objeto inanimado?'), additional_kwargs={})]) middle=[] last=Ollama(model='minimax-m2.5:cloud', temperature=0.1)
first=ChatPromptTemplate(in

In [ ]:
from pydantic import BaseModel
from typing import Literal


class Categoria(BaseModel):
    """Cria a categoria correspondente a pergunta feita"""
    area_conhecimento: Literal["Matematica", "Portugues", "Biologia", "Nao sei"]

In [7]:
from langchain_core.output_parsers import PydanticOutputParser


model_estruturado = PydanticOutputParser(pydantic_object=Categoria)

In [8]:
duvida_usuario = "Como fazer o cálculo do teorema de pitágoras?"
prompt_classificacao = ChatPromptTemplate.from_template(
    "Avalie a seguinte pergunta e retorne um JSON exato sobre qual área de conhecimento ela pertence.\\n\\n"
    "Pergunta: {duvida}\\n\\n"
    "{instructions}"
).partial(instructions=model_estruturado.get_format_instructions())

chain_model_estruturado = prompt_classificacao | chat_models | model_estruturado

In [ ]:
classificacao = chain_model_estruturado.invoke({"duvida": duvida_usuario})

classificacao.area_conhecimento

NameError: name 'chain_model_estrutudo' is not defined

In [ ]:
classificacao = chain_model_estruturado.invoke({"duvida": "Quantas celulas existe no corpo humano?"})

classificacao.area_conhecimento

'Biologia'

In [ ]:
classificacao = chain_model_estruturado.invoke({"duvida": "Quantas palavras existe no alfabeto latim?"})

classificacao.area_conhecimento

'Portugues'

In [ ]:
classificacao = chain_model_estruturado.invoke({"duvida": "Que ano o homem pisou na lua?"})

classificacao.area_conhecimento

'Nao sei'

In [ ]:
classificacao = chain_model_estruturado.invoke({"duvida": "Com quantos anos uma menina se torna mulher?"})

classificacao.area_conhecimento

'Biologia'

In [ ]:
math_questions = [
    "Quanto é 10/2?",
    "Quanto é 15 x 15?",
    "Qual a raiz quadrada de 144?",
    "Quanto é 100 + 250?",
    "Qual o valor de 3²?"
]

biology_questions = [
    "Qual a função do coração no corpo humano?",
    "O que é fotossíntese?",
    "Quantos ossos tem o corpo humano adulto?",
    "Qual a fórmula da água?",
    "O que são células?"
]

portuguese_questions = [
    "Qual o antônimo de 'feliz'?",
    "Qual a diferença entre 'seu' e 'suas'?",
    "Qual a regra de acentuação de 'público'?",
    "Qual o plural de 'cidadão'?",
    "O que é um substantivo?"
]

questions = [
    "Quanto é 10/2?",
    "Qual a função do coração no corpo humano?",
    "Qual é o antônimo de 'feliz'?",
    "Quanto é 15 x 15?",
    "O que é fotossíntese?",
    "Qual a diferença entre 'seu' e 'suas'?",
    "Qual a raiz quadrada de 144?",
    "Quantos ossos tem o corpo humano adulto?",
    "Qual a regra de acentuação de 'público'?",
    "Qual a fórmula da água?"
]

In [ ]:
for question in math_questions + biology_questions + portuguese_questions + questions:
    classificacao = chain_model_estruturado.invoke({"duvida": question})
    print(question, classificacao.area_conhecimento)

KeyboardInterrupt: 